In [1]:
# Start from scratch!
# Don't use my package to do this!
from robustraster import dask_cluster_manager

json_key = r"R:\SCRATCH\adrianom\credentials\earthengineapi\robust-raster-cecdcc4b5fba.json"
dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="full", threads_per_worker=1, n_workers=2, memory_limit="5GB")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [2]:
# 2. Authenticate Google Earth Engine on all Dask workers.
from robustraster import dask_plugins

ee_plugin = dask_plugins.EEPlugin(json_key)
dask_client = dask_cluster.get_dask_client
dask_client.register_plugin(ee_plugin)

{'tcp://127.0.0.1:58490': {'status': 'OK'},
 'tcp://127.0.0.1:58493': {'status': 'OK'}}

In [27]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.

from robustraster import dataset_manager
import ee
import json
import xarray as xr
from dask.distributed import performance_report
import dask

# Although we authenticated Google Earth Engine on our Dask workers, we also need to authenticate
# Google Earth Engine on our local machine!
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

WSDemo = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
California = ee.FeatureCollection("projects/robust-raster/assets/boundaries/California")

#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2014-01-01', '2014-12-31')
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2020-01-01', '2020-01-02').map(prep_sr_l8).select(['SR_B4', 'SR_B5']).sort("system:time_start")
ds = xr.open_dataset(ic,
            engine='ee', 
            crs="EPSG:3310", 
            scale=1000,
            geometry=WSDemo.geometry(),
            chunks={"time": 48, "X": 512, "Y": 256})

In [28]:
print(ds)

<xarray.Dataset> Size: 55MB
Dimensions:  (time: 302, X: 138, Y: 164)
Coordinates:
  * time     (time) datetime64[ns] 2kB 2020-01-01T01:07:28.840000 ... 2020-01...
  * X        (X) float64 1kB -7.664e+04 -7.654e+04 ... -6.304e+04 -6.294e+04
  * Y        (Y) float64 1kB 1.886e+05 1.887e+05 ... 2.048e+05 2.049e+05
Data variables:
    SR_B4    (time, X, Y) float32 27MB dask.array<chunksize=(48, 138, 164), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 27MB dask.array<chunksize=(48, 138, 164), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_

In [29]:
import dask.array as da
import os
import rioxarray
from dask.distributed import get_worker

def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df

def user_function_wrapper(ds, user_func, *args, **kwargs):
    worker = get_worker()
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    print("THATS NOT THE SHAPE OF MY HEART!")
    #for i, (label, chunk) in enumerate(ds_output.items()):
    #    for var in ds_output.data_vars:
    #        print(chunk[var])
    #        chunk[var].rio.to_raster(f"{var}_tile_{i}.tif")
    return ds_output


def generate_template_xarray(ds, user_func):
    # Dynamically determine dimension names
    dim_names = list(ds.sizes.keys())
        
    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)
        
    # Apply the processing function to this chunk
    processed_chunk = user_func(one_chunk)
        
    # Create the template using a combination of original data variables and newly created ones
    template_vars = {}
        
    for var in processed_chunk.data_vars:
        if var in ds.data_vars:
            # Use the original dataset's shape and chunking for existing variables
            template_vars[var] = (processed_chunk[var].dims, 
                                        da.empty(ds[var].shape, 
                                        chunks=ds[var].chunks, 
                                        dtype=processed_chunk[var].dtype))
        else:
            # For new variables, define the shape and chunks manually based on the original chunking strategy
            new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)
            new_var_chunks = tuple(ds.chunks[dim][0] for dim in processed_chunk[var].dims)
            template_vars[var] = (processed_chunk[var].dims, 
                                        da.empty(new_var_shape, 
                                        chunks=new_var_chunks, 
                                        dtype=processed_chunk[var].dtype))
        
    template = xr.Dataset(
        template_vars,
        coords={coord: ds.coords[coord] for coord in ds.coords},
        attrs=ds.attrs
    )
        
    return template

In [ ]:
# EXPORT EACH DATA VARIABLE CHUNK AS IT'S OWN GEOTIFF

template_xarray = generate_template_xarray(ds, compute_ndvi)
ds.chunk({"time": 48, "X": 512, "Y": 256})
result = xr.map_blocks(user_function_wrapper, 
                        ds, 
                        args=(compute_ndvi,),
                        template=template_xarray)


ds_renamed = result.rename({'X': 'x', 'Y': 'y'})
ds_transposed = ds_renamed.transpose('time', 'y', 'x').rio.write_crs("EPSG:3310")
for index_value in ds_transposed["time"].values:
        # Select the slice of data for the current index
        ds_slice = ds_transposed.sel({"time": index_value})

        # Convert the index value to a string for file naming
        index_str = str(index_value).replace(":", "").replace("-", "").replace(" ", "_")

        for var in ds_slice.data_vars:
            data_array = ds_slice[var]
            print(data_array)
            # Ensure the DataArray has CRS metadata before exporting
            if "spatial_ref" not in data_array.coords:
                data_array = data_array.rio.write_crs("EPSG:4326")  # Change EPSG if needed

            # Define output file path
            output_path = os.path.join(f"{var}_{index_str}.tif")

            # Export as a single-band GeoTIFF
            data_array.rio.to_raster(output_path)

            print(f"Exported: {output_path}") 

In [32]:
template_xarray = generate_template_xarray(ds, compute_ndvi)
ds.chunk({"time": 48, "X": 512, "Y": 256})
result = xr.map_blocks(user_function_wrapper, 
                        ds, 
                        args=(compute_ndvi,),
                        template=template_xarray)


ds_renamed = result.rename({'X': 'x', 'Y': 'y'})
ds_transposed = ds_renamed.transpose('time', 'y', 'x').rio.write_crs("EPSG:3310")
output_dir = "tiles"
os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists

for index_value in ds_transposed["time"].values:
    # Select the slice of data for the current time index
    ds_slice = ds_transposed.sel({"time": index_value})

    # Convert time index to string for filename
    index_str = str(index_value).replace(":", "").replace("-", "").replace(" ", "_")

    # Convert Dataset to a DataArray with a new "band" dimension
    stacked = ds_slice.to_array(dim="band")

    # Ensure CRS is set
    if "spatial_ref" not in stacked.coords:
        stacked = stacked.rio.write_crs("EPSG:3310")  # Change EPSG as needed

    # Define output path for this multi-band raster
    output_path = os.path.join(output_dir, f"multi_band_{index_str}.tif")

    # Export as a multi-band GeoTIFF
    stacked.rio.to_raster(output_path, driver="GTiff")

    print(f"Exported: {output_path}")

R:\Users\adrianom.UNR\AppData\Local\Temp\19\ipykernel_79632\4174374160.py:48: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)


Exported: tiles\multi_band_20200101T010728.840000000.tif
Exported: tiles\multi_band_20200101T010752.718000000.tif
Exported: tiles\multi_band_20200101T010816.601000000.tif
Exported: tiles\multi_band_20200101T010840.487000000.tif
Exported: tiles\multi_band_20200101T010904.374000000.tif
Exported: tiles\multi_band_20200101T010928.257000000.tif
Exported: tiles\multi_band_20200101T010952.144000000.tif
Exported: tiles\multi_band_20200101T011127.704000000.tif
Exported: tiles\multi_band_20200101T012013.594000000.tif
Exported: tiles\multi_band_20200101T012037.511000000.tif
Exported: tiles\multi_band_20200101T012101.427000000.tif
Exported: tiles\multi_band_20200101T012125.352000000.tif
Exported: tiles\multi_band_20200101T012149.277000000.tif
Exported: tiles\multi_band_20200101T012213.206000000.tif
Exported: tiles\multi_band_20200101T012301.056000000.tif
Exported: tiles\multi_band_20200101T012324.985000000.tif
Exported: tiles\multi_band_20200101T012348.919000000.tif
Exported: tiles\multi_band_2020

KeyboardInterrupt: 

In [37]:
template_xarray = generate_template_xarray(ds, compute_ndvi)
ds.chunk({"time": 48, "X": 512, "Y": 256})
result = xr.map_blocks(user_function_wrapper, 
                        ds, 
                        args=(compute_ndvi,),
                        template=template_xarray)


ds_renamed = result.rename({'X': 'x', 'Y': 'y'})
ds_transposed = ds_renamed.transpose('time', 'y', 'x').rio.write_crs("EPSG:3310")
output_dir = "tiles"

os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists

for index_value in ds_transposed["time"].values:
    # Select the slice of data for the current time index
    ds_slice = ds_transposed.sel({"time": index_value})

    # Convert time index to string for filename
    index_str = str(index_value).replace(":", "").replace("-", "").replace(" ", "_")

    # Convert Dataset to a DataArray with a new "band" dimension
    stacked = ds_slice.to_array(dim="band")

    # Assign band names using data variable names
    stacked = stacked.assign_coords(band=list(ds_slice.data_vars))

    print(stacked)
    # Ensure CRS is set
    if "spatial_ref" not in stacked.coords:
        stacked = stacked.rio.write_crs("EPSG:3310")  # Change EPSG as needed

    # Define output path for this multi-band raster
    output_path = os.path.join(output_dir, f"multi_band_{index_str}.tif")

    # Export as a multi-band GeoTIFF
    stacked.rio.to_raster(output_path, driver="GTiff")

    print(f"Exported: {output_path} with bands {list(ds_slice.data_vars)}")

R:\Users\adrianom.UNR\AppData\Local\Temp\19\ipykernel_79632\4174374160.py:48: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)


<xarray.DataArray (band: 3, y: 164, x: 138)> Size: 272kB
dask.array<stack, shape=(3, 164, 138), dtype=float32, chunksize=(1, 164, 138), chunktype=numpy.ndarray>
Coordinates:
    time         datetime64[ns] 8B 2020-01-01T01:07:28.840000
  * x            (x) float64 1kB -7.664e+04 -7.654e+04 ... -6.304e+04 -6.294e+04
  * y            (y) float64 1kB 1.886e+05 1.887e+05 ... 2.048e+05 2.049e+05
    spatial_ref  int64 8B 0
  * band         (band) <U5 60B 'SR_B5' 'SR_B4' 'ndvi'
Attributes: (12/25)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_min:    0.0
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:

KeyboardInterrupt: 

In [35]:
import rasterio
# Open the raster file
with rasterio.open("./tiles/multi_band_20200101T010728.840000000.tif") as src:
    # Print band count
    print(f"Number of bands: {src.count}")

    # Print band indexes (rasterio doesn't store band names explicitly)
    print(f"Band indexes: {src.indexes}")

    # Print metadata to check if band descriptions are available
    print(f"Metadata: {src.meta}")

    # If the raster has band descriptions
    band_names = src.descriptions  # This returns a tuple of band descriptions
    print(f"Band names: {band_names}")


Number of bands: 3
Band indexes: (1, 2, 3)
Metadata: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': None, 'width': 138, 'height': 164, 'count': 3, 'crs': CRS.from_epsg(3310), 'transform': Affine(100.0, 0.0, -76694.52679078851,
       0.0, 100.0, 188585.21294134157)}
Band names: (None, None, None)
